### Задание

1. Выбрать QA датасет (https://huggingface.co/datasets). 
2. На выбранном датасете обучить модель https://huggingface.co/intfloat/multilingual-e5-small с помощью этого кода.
3. Переписать код под актуальную версию sentence-transformers.
4. Сравнить результаты

In [ ]:
# pip install transformers==4.45.2 
# pip install sentence-transformers==3.1.1

In [ ]:
pip install sentence-transformers

In [6]:
# import pickle
# from random import shuffle

# from datasets import load_dataset
# from sentence_transformers import InputExample

import os
import random

import numpy as np
import torch

from datasets import load_dataset, Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    util,
)

/tmp/ipykernel_3201605/3334920017.py:15: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


In [5]:
import torch

device = 'mps' if torch.backends.mps.is_built() else 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

cuda


In [6]:
# from sentence_transformers import SentenceTransformer, models


# def raw_bi_encoder():
#     word_embedding_model = models.Transformer('cointegrated/rubert-tiny2', max_seq_length=256)
#     pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())

#     bi_encoder = SentenceTransformer(
#         modules=[word_embedding_model, pooling_model],
#         device=device,
#     )

#     return bi_encoder

def raw_bi_encoder(model_name: str, max_seq_length: int = 512):
    model = SentenceTransformer(
        model_name,
        device=device,
    )

    model.max_seq_length = max_seq_length

    return model

In [10]:
# bi_encoder = raw_bi_encoder()

# print(bi_encoder)
MODEL_NAME = "intfloat/multilingual-e5-small"
bi_encoder = raw_bi_encoder(MODEL_NAME, max_seq_length=512)

print(bi_encoder)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4265.92it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


## Служебные функции для работы с данными

In [3]:
DATASET_ID = "kuznetsoffandrey/sberquad"
SEED = 42

# Префиксы для multilingual-e5-small.
# для rubert нужны пустые префиксы: "" и "".
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

# Простой кэш исходного датасета в памяти, чтобы не грузить его каждый раз.
_RAW_DATASET_CACHE = {}

In [4]:
def shrink_repeated_samples(
        queries,
        docs,
        labels,
):
    docs_was = set()
    qs = []
    ds = []
    ls = []
    for i in range(len(queries)):
        q, d, l = queries[i], docs[i], labels[i]
        if d in docs_was:
            continue
        qs.append(q)
        ds.append(d)
        ls.append(l)
        docs_was.add(d)

    return qs, ds, ls


# def trainval_shuffled_data(from_file=True):
#     datafile = 'trainval.pkl'

#     if from_file:
#         with open(datafile, 'rb') as f:
#             data = pickle.load(f)
#         return data

#     good, bad = load_trainval_samples()
#     data = good + bad
#     shuffle(data)

#     with open(datafile, 'wb') as f:
#         pickle.dump(data, f)

#     return data
def trainval_shuffled_data(from_file=None):
    """
    Возвращает перемешанные обучающие пары:
        (question, context, label)

    from_file оставлен для совместимости со старыми вызовами,
    но больше не используется.
    """
    good, bad = load_trainval_samples()
    data = good + bad

    random.Random(SEED).shuffle(data)

    return data

# def test_shuffled_data(from_file=True):
#     datafile = 'test.pkl'

#     if from_file:
#         with open(datafile, 'rb') as f:
#             data = pickle.load(f)
#         return data

#     good, bad = load_test_samples()
#     data = good + bad

#     with open(datafile, 'wb') as f:
#         pickle.dump(data, f)

#     return data
def test_shuffled_data(from_file=None):
    """
    Возвращает перемешанные тестовые пары:
        (question, context, label)

    from_file оставлен для совместимости со старыми вызовами,
    но больше не используется.
    """
    good, bad = load_test_samples()
    data = good + bad

    random.Random(SEED + 1).shuffle(data)

    return data

# def trainval_examples():
#     data = trainval_shuffled_data(from_file=True)

#     trainval = []
#     for question, document, cos in data:
#         trainval.append(
#             InputExample(texts=[question, document], label=cos)
#         )

#     return trainval
def trainval_examples(from_file=None):
    """
    Старая функция возвращала InputExample(...).

    Новая версия возвращает обычный список словарей:
        [{"anchor": ..., "positive": ..., "score": ...}, ...]
    """
    data = trainval_shuffled_data()

    return pairs_to_rows(
        pairs=data,
        query_prefix=QUERY_PREFIX,
        passage_prefix=PASSAGE_PREFIX,
    )

def load_test_samples():
    _, _, test = _load_sberquad_dataset()

    questions = test['question'].tolist()
    contexts = test['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def load_trainval_samples():
    train, val, _ = _load_sberquad_dataset()

    questions = train['question'].tolist() + val['question'].tolist()
    contexts = train['context'].tolist() + val['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def _inflate_with_negative_samples(
        questions,
        contexts,
        delta=30,  # in sberquad dataset same contexts are close
):
    good_q_c_cos = list(zip(questions, contexts, [1.0] * len(questions)))

    bad_q_c_cos = []
    n = len(good_q_c_cos)
    for i in range(n):
        cur_q, cur_c, _ = good_q_c_cos[i]
        next_q, next_c, _ = good_q_c_cos[(i + delta) % n]
        if next_c != cur_c:
            bad_q_c_cos.append((cur_q, next_c, 0.0))

    return good_q_c_cos, bad_q_c_cos


# def _load_raw_sberquad_dataset(from_file=True):
#     sberquad = 'sberquad.pkl'
#     if from_file:
#         with open(sberquad, 'rb') as f:
#             df = pickle.load(f)
#     else:
#         df = load_dataset('sberquad')
#         with open(sberquad, 'wb') as f:
#             pickle.dump(df, f)

#     return df
def _load_raw_sberquad_dataset(from_file=None, dataset_id=DATASET_ID):
    """
    Загружаем датасет и кэшируем его в памяти.

    from_file оставлен в сигнатуре для совместимости со старым кодом,
    но больше не используется: pickle-файлы мы не читаем.
    """
    if dataset_id not in _RAW_DATASET_CACHE:
        _RAW_DATASET_CACHE[dataset_id] = load_dataset(dataset_id)

    return _RAW_DATASET_CACHE[dataset_id]

# def _load_sberquad_dataset():
#     df = _load_raw_sberquad_dataset()
#     train = df['train'].to_pandas()
#     validation = df['validation'].to_pandas()
#     test = df['test'].to_pandas()

#     return train, validation, test
def _load_sberquad_dataset():
    """
    Возвращаем train, validation, test в виде pandas DataFrame,
    чтобы старые функции ниже могли работать без изменений.
    """
    ds = _load_raw_sberquad_dataset()

    if "train" not in ds:
        raise ValueError(f"В датасете {DATASET_ID} нет сплита train.")

    train_split = ds["train"]

    if "validation" in ds:
        validation_split = ds["validation"]
    else:
        validation_split = train_split

    if "test" in ds:
        test_split = ds["test"]
    elif "validation" in ds:
        test_split = ds["validation"]
    else:
        test_split = train_split

    train = train_split.to_pandas()

    if "validation" in ds:
        validation = validation_split.to_pandas()
    else:
        validation = train.copy()

    if "test" in ds:
        test = test_split.to_pandas()
    elif "validation" in ds:
        test = validation.copy()
    else:
        test = train.copy()

    return train, validation, test

def pairs_to_rows(pairs, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """
    Превращает:
        [(question, context, label), ...]

    в список словарей:
        {
            "anchor": "...",
            "positive": "...",
            "score": float
        }

    Такой формат понимают современные loss functions в sentence-transformers.
    """
    rows = []

    for question, context, score in pairs:
        rows.append(
            {
                "anchor": f"{query_prefix}{question}",
                "positive": f"{passage_prefix}{context}",
                "score": float(score),
            }
        )

    return rows

def trainval_dataset(from_file=None):
    """
    Готовит Hugging Face Dataset из обучающих пар.
    """
    rows = trainval_examples()

    return Dataset.from_list(rows)

# Метрики близости без дообучения на trainval

In [16]:
# trainval = trainval_shuffled_data(from_file=True)

# n_samples = 1000
# # questions, document_contexts, coss = zip(*trainval[:n_samples])
# questions, document_contexts, coss = zip(*trainval)

trainval = trainval_shuffled_data()

n_samples = 1000

if len(trainval) > n_samples:
    trainval_sample = random.Random(SEED).sample(trainval, n_samples)
else:
    trainval_sample = trainval

questions_raw, document_contexts_raw, coss = zip(*trainval_sample)

# e5-small требует query/passage, rubert-tiny2 — нет.
if "multilingual-e5" in MODEL_NAME:
    questions = [QUERY_PREFIX + q for q in questions_raw]
    document_contexts = [PASSAGE_PREFIX + d for d in document_contexts_raw]
else:
    questions = questions_raw
    document_contexts = document_contexts_raw

Generating test split: 100%|██████████| 23936/23936 [00:00<00:00, 274132.88 examples/s]


In [ ]:
# evaluator = evaluation.EmbeddingSimilarityEvaluator(questions, document_contexts, coss)

In [18]:
# bi_encoder.evaluate(evaluator)

from sentence_transformers import evaluation

evaluator = evaluation.EmbeddingSimilarityEvaluator(
    questions,
    document_contexts,
    coss,
)

evaluator(bi_encoder)

/tmp/ipykernel_3130986/3565600120.py:3: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import evaluation


{'pearson_cosine': 0.9152786941445872, 'spearman_cosine': 0.8628457645117781}

# Метрики близости без дообучения на test

In [ ]:
# n_test_samples = 1000
# test = test_shuffled_data(from_file=True)
# test_questions, test_document_contexts, test_coss = zip(*test[:n_test_samples])

In [ ]:
# evaluator = evaluation.EmbeddingSimilarityEvaluator(test_questions, test_document_contexts, test_coss)

In [ ]:
# bi_encoder = raw_bi_encoder()

In [20]:
# bi_encoder.evaluate(evaluator)
MAX_LEN = 512

n_test_samples = 1000

test = test_shuffled_data()

test_sample = test[:n_test_samples]

test_questions_raw, test_document_contexts_raw, test_coss = zip(*test_sample)

# Для multilingual-e5-small добавляем query/passage.
# Для rubert-tiny2 оставляем без префиксов.
if "multilingual-e5" in MODEL_NAME:
    test_questions = [QUERY_PREFIX + q for q in test_questions_raw]
    test_document_contexts = [PASSAGE_PREFIX + d for d in test_document_contexts_raw]
else:
    test_questions = test_questions_raw
    test_document_contexts = test_document_contexts_raw


evaluator_test = evaluation.EmbeddingSimilarityEvaluator(
    test_questions,
    test_document_contexts,
    test_coss,
)

bi_encoder = raw_bi_encoder(MODEL_NAME, max_seq_length=MAX_LEN)

baseline_test_metrics = evaluator_test(bi_encoder)

baseline_test_metrics

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4859.21it/s]


{'pearson_cosine': 0.9201645294546714, 'spearman_cosine': 0.8646678652730023}

# Дообучение

In [ ]:
# # trainval_examples = trainval_examples()[:n_samples]
# trainval_examples = trainval_examples()
# print(trainval_examples[0].__dict__)

In [ ]:
# from torch.utils.data import DataLoader

# trainval_dataloader_set = DataLoader(trainval_examples, shuffle=True, batch_size=32,
#                                      collate_fn=bi_encoder.smart_batching_collate)

In [ ]:
# from sentence_transformers import losses

# trainval_loss_set = losses.CosineSimilarityLoss(bi_encoder)

In [ ]:
# trainval_loss_set

In [ ]:
# (query_batch, context_batch), labels = next(iter(trainval_dataloader_set))
# query_batch['input_ids'].shape, context_batch['input_ids'].shape, labels.shape

In [ ]:
# bi_encoder.fit(
#     train_objectives=[(trainval_dataloader_set, trainval_loss_set)],
#     output_path='qa/results',
#     epochs=5,
#     evaluator=evaluator,
# )

In [21]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

# Настройки обучения
OUTPUT_DIR = "results/e5_sberquad_finetuned"
EPOCHS = 2
TRAIN_BATCH_SIZE = 16
GRAD_ACC_STEPS = 1
LEARNING_RATE = 2e-5

# Убедимся, что префиксы соответствуют выбранной модели
if "multilingual-e5" in MODEL_NAME:
    QUERY_PREFIX = "query: "
    PASSAGE_PREFIX = "passage: "
else:
    QUERY_PREFIX = ""
    PASSAGE_PREFIX = ""

# Готовим Dataset для обучения.
# Эта функция использует trainval_shuffled_data(),
# превращает пары (question, context, label) в Dataset
# с колонками anchor / positive / score.
train_dataset = trainval_dataset()

print("Первый обучающий пример:")
print(train_dataset[0])
print()
print("Размер обучающего датасета:")
print(len(train_dataset))


# Для CosineSimilarityLoss dataset должен иметь колонки:
#   anchor
#   positive
#   score
train_loss = losses.CosineSimilarityLoss(bi_encoder)

training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    logging_steps=20,

    # Для нашего примера не обязательно хранить много чекпоинтов.
    save_strategy="no",

    # Для стабильности используем fp32.
    # Если GPU поддерживает bf16 и нет проблем, можно включить bf16=True.
    fp16=False,
    bf16=False,

    report_to=[],
    seed=SEED,

    # Важно для обучения через Dataset с колонками anchor / positive / score.
    remove_unused_columns=False,

    # в VSCode часто безопаснее оставить 0.
    dataloader_num_workers=0,
)

trainer = SentenceTransformerTrainer(
    model=bi_encoder,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()

# Сохраняем дообученную модель
bi_encoder.save_pretrained(OUTPUT_DIR)

print(f"\nОбученная модель сохранена в {OUTPUT_DIR}")

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Первый обучающий пример:
{'anchor': 'query: Когда был зарегистрирован Коммерческий банк „БНП Париба Восток“ ?', 'positive': 'passage: Критерии, используемые Трубецким, были весьма близки тем основанным на дистрибуции методам, которые в то время активно разрабатывались в американском дескриптивизме, в работах Леонарда Блумфилда, Морриса Сводеша и других. К структуралистам был отчасти близок по своим взглядам Эдвард Сепир. В частности, в известной работе Sound patterns in language [6] он подчеркивал, что лингвистическая значимость артикуляционных событий обусловлена не их физической природой, а тем, как они соотносятся с другими событиями в системе данного языка: так, звук, производимый при задувании свечи, с акустической точки зрения похож на звук, фигурирующий в ряде разновидностей английского языка в словах which или white ([ʍ]), однако лингвистическая их значимость совершенно разная.', 'score': 0.0}

Размер обучающего датасета:
100728


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
20,0.315844
40,0.274489
60,0.311383
80,0.267396
100,0.264009
120,0.234684
140,0.176577
160,0.144132
180,0.110187
200,0.101343


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]



Обученная модель сохранена в results/e5_sberquad_finetuned


In [22]:
fine_tuned_test_metrics = evaluator_test(bi_encoder)

print("Baseline:")
print(baseline_test_metrics)

print()
print("Fine-tuned:")
print(fine_tuned_test_metrics)

Baseline:
{'pearson_cosine': 0.9201645294546714, 'spearman_cosine': 0.8646678652730023}

Fine-tuned:
{'pearson_cosine': 0.9129187057174923, 'spearman_cosine': 0.8514345670144171}


# Загрузка чекпоинта

In [23]:
# from sentence_transformers import SentenceTransformer

# finetuned_bi_encoder = SentenceTransformer('qa/results')

In [24]:
# finetuned_bi_encoder.evaluate(evaluator)

In [25]:
from sentence_transformers import SentenceTransformer

# Если переменная OUTPUT_DIR уже определена, используем её.
# Иначе ставим путь по умолчанию, который мы использовали при обучении.
OUTPUT_DIR = globals().get("OUTPUT_DIR", "results/e5_sberquad_finetuned")
DEVICE = globals().get("DEVICE", "cpu")
MAX_LEN = globals().get("MAX_LEN", 512)

finetuned_bi_encoder = SentenceTransformer(
    OUTPUT_DIR,
    device=DEVICE,
)

finetuned_bi_encoder.max_seq_length = MAX_LEN

loaded_fine_tuned_test_metrics = evaluator_test(finetuned_bi_encoder)

loaded_fine_tuned_test_metrics

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2974.79it/s]


{'pearson_cosine': 0.9129187049933101, 'spearman_cosine': 0.8514345670144171}

# Расчёт метрик

In [26]:
# data = test_shuffled_data(from_file=True)

In [27]:
# n_test_samples = 1000
# queries, docs, labels = zip(*data[:n_test_samples])

In [28]:
# print('docs count before shrinking: ', len(docs))

In [29]:
# docs = list(set(docs))

In [30]:
# print('docs count without duplicates: ', len(docs))

In [31]:
# doc_embs = finetuned_bi_encoder.encode(
#     docs,
#     convert_to_tensor=True,
#     show_progress_bar=True,
# )

In [32]:
# query_embs = finetuned_bi_encoder.encode(
#     queries,
#     convert_to_tensor=True,
#     show_progress_bar=True,
# )

In [33]:
# from sentence_transformers import util

In [34]:
# cos_sim_score = util.semantic_search(query_embs, doc_embs, top_k=3)

In [35]:
# for_first_query = cos_sim_score[0]
# print(for_first_query)

In [36]:
# print('Question: ', queries[0])
# print()
# for no, ir in enumerate(cos_sim_score[0]):
#     corpus_id = ir["corpus_id"]
#     print('corpus id: ', corpus_id)
#     print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
#     print()

In [37]:
# data = test_shuffled_data(from_file=True)
# n_test_samples = 1000
# queries, docs, labels = zip(*data[:n_test_samples])

# queries, docs, labels = shrink_repeated_samples(
#     queries=queries,
#     docs=docs,
#     labels=labels,
# )
# print(len(docs))

In [39]:
from sentence_transformers import util

model_for_retrieval = finetuned_bi_encoder

def encode_with_prefix(model, texts, prefix):
    """
    Кодирование текстов с префиксом.

    Для multilingual-e5-small prefix должен быть:
        query:
        passage:

    Для rubert-tiny2 prefix должен быть пустой:
        ""
    """
    texts = [prefix + str(text) for text in texts]

    embs = model.encode(
        texts,
        convert_to_tensor=True,
        device=DEVICE,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    return embs


# Готовим тестовые пары
test = test_shuffled_data()

n_test_samples = 1000
test_sample = test[:n_test_samples]

queries, docs, labels = zip(*test_sample)

print("docs count before shrinking:", len(docs))

# shrink_repeated_samples должен вызываться ДО encode.
# Именно он убирает дубликаты документов и сохраняет alignment:
# queries[i] <-> docs[i]
queries, docs, labels = shrink_repeated_samples(
    queries=queries,
    docs=docs,
    labels=labels,
)

print("docs count after shrinking:", len(docs))
print("queries count after shrinking:", len(queries))


# Для multilingual-e5-small используем query/passage
if "multilingual-e5" in MODEL_NAME:
    query_prefix = QUERY_PREFIX
    doc_prefix = PASSAGE_PREFIX
else:
    query_prefix = ""
    doc_prefix = ""


query_embs = encode_with_prefix(
    model=model_for_retrieval,
    texts=queries,
    prefix=query_prefix,
)

doc_embs = encode_with_prefix(
    model=model_for_retrieval,
    texts=docs,
    prefix=doc_prefix,
)


# Визуализация top-k для первого вопроса
top_k_vis = min(3, len(docs))

cos_sim_score = util.semantic_search(
    query_embs,
    doc_embs,
    top_k=top_k_vis,
)

print()
print("Example query:", queries[0])
print()

for no, ir in enumerate(cos_sim_score[0]):
    corpus_id = int(ir["corpus_id"])
    score = float(ir["score"])

    print(f"Document {no + 1}: corpus_id={corpus_id}, cosine={score:.3f}")
    print()
    print(docs[corpus_id])
    print()

docs count before shrinking: 1000
docs count after shrinking: 886
queries count after shrinking: 886

Example query: На каких линиях ось пути и линия трассы совпадают?

Document 1: corpus_id=844, cosine=0.882

На железных дорогах СНГ платформы различаются по высоте на высокие и низкие. Высокие платформы преимущественно стоят на Московской железной дороге и Октябрьской железной дороге, где обслуживают не только поезда дальнего следования, но и электропоезда. На других крупных вокзалах обычно стоят высокие платформы исключительно для поездов дальнего следования; для электропоездов стоят низкие платформы. На переменных (переменным током) и неэлектрифицированных линиях пассажирские платформы обычно низкие.

Document 2: corpus_id=306, cosine=0.775

В сутки по дороге курсируют более 500 пассажирских, 3000 пригородных и 2500 грузовых поездов. Эксплуатационная длина главных путей МЖД составляет 8,8 тыс. км, из них электрифицировано 4,3 тыс. км (или 48%). Развёрнутая длина — более 13 тыс. км. В

# Получение предсказаний модели и меток

In [ ]:
# doc_embs = finetuned_bi_encoder.encode(
#     docs,
#     convert_to_tensor=True,
#     show_progress_bar=True,
# )

In [ ]:
# query_embs = finetuned_bi_encoder.encode(
#     queries,
#     convert_to_tensor=True,
#     show_progress_bar=True,
# )

In [ ]:
# cos_scores = util.semantic_search(query_embs, doc_embs, top_k=100)

In [ ]:
# print(len(cos_scores))

In [ ]:
# print(cos_scores[0][:5])

In [ ]:
# top_k = 5

In [ ]:
# print('Question: ', queries[0])
# print()
# for no, ir in enumerate(cos_scores[0][:top_k]):
#     corpus_id = ir["corpus_id"]
#     print('corpus id: ', corpus_id)
#     print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
#     print()

In [ ]:
# print(docs[582])

In [ ]:
# n = len(queries)
# print(n)

In [40]:
# queries и docs уже должны быть подготовлены через shrink_repeated_samples(...),
# чтобы queries[i] соответствовал docs[i].
n_docs = len(docs)
n_queries = len(queries)

print("queries count:", n_queries)
print("docs count:", n_docs)

top_k_scores = min(100, n_docs)
top_k_vis = min(5, n_docs)


doc_embs = encode_with_prefix(
    model=model_for_retrieval,
    texts=docs,
    prefix=doc_prefix,
)

query_embs = encode_with_prefix(
    model=model_for_retrieval,
    texts=queries,
    prefix=query_prefix,
)


cos_scores = util.semantic_search(
    query_embs,
    doc_embs,
    top_k=top_k_scores,
)


print()
print("Number of scored queries:", len(cos_scores))
print("First query top-5:", cos_scores[0][:5])


print()
print("Example query:", queries[0])
print()

for no, ir in enumerate(cos_scores[0][:top_k_vis]):
    corpus_id = int(ir["corpus_id"])
    score = float(ir["score"])

    print(f"Document {no + 1}: corpus_id={corpus_id}, cosine={score:.3f}")
    print()
    print(docs[corpus_id])
    print()


# Старая проверка docs[582] была небезопасной.
# Оставим её с проверкой границ, если хочется что-то посмотреть.
example_doc_index = 582

if len(docs) > example_doc_index:
    print("Example document by fixed index:")
    print(docs[example_doc_index])
else:
    print(
        f"В shrunk docs только {len(docs)} документов, "
        f"индекс {example_doc_index} недоступен."
    )


queries count: 886
docs count: 886

Number of scored queries: 886
First query top-5: [{'corpus_id': 844, 'score': 0.88191819190979}, {'corpus_id': 306, 'score': 0.7750690579414368}, {'corpus_id': 819, 'score': 0.7461245059967041}, {'corpus_id': 465, 'score': 0.7422165870666504}, {'corpus_id': 485, 'score': 0.7255663275718689}]

Example query: На каких линиях ось пути и линия трассы совпадают?

Document 1: corpus_id=844, cosine=0.882

На железных дорогах СНГ платформы различаются по высоте на высокие и низкие. Высокие платформы преимущественно стоят на Московской железной дороге и Октябрьской железной дороге, где обслуживают не только поезда дальнего следования, но и электропоезда. На других крупных вокзалах обычно стоят высокие платформы исключительно для поездов дальнего следования; для электропоездов стоят низкие платформы. На переменных (переменным током) и неэлектрифицированных линиях пассажирские платформы обычно низкие.

Document 2: corpus_id=306, cosine=0.775

В сутки по дороге 

In [41]:
def build_target_mask_for_i(i, n):
    target_mask = [False] * n
    target_mask[i] = True

    return target_mask

In [42]:
def build_pred_mask_for_i(
        cos_scores,
        i,
        n,
):
    pred_mask = [0.0] * n
    qi_scores = cos_scores[i]

    for docid_score in qi_scores:
        doc_id = docid_score['corpus_id']
        score = docid_score['score']
        pred_mask[int(doc_id)] = score

    return pred_mask

# Расчёт метрик: Recall@5, MRR, MAP, NDCG@10

In [43]:
# from statistics import mean
# from torchmetrics.functional.retrieval import retrieval_reciprocal_rank, retrieval_average_precision, \
#     retrieval_normalized_dcg, retrieval_recall

# r5s = []
# mrrs = []
# rmaps = []
# ndcgs = []
# for i in range(n):
#     pred = build_pred_mask_for_i(cos_scores, i=i, n=n)
#     target = build_target_mask_for_i(i=i, n=n)

#     pred = torch.tensor(pred)
#     target = torch.tensor(target)
    
#     r5 = retrieval_recall(preds=pred, target=target, top_k=5).item()
#     mrr = retrieval_reciprocal_rank(preds=pred, target=target, top_k=5).item()
#     rmap = retrieval_average_precision(preds=pred, target=target, top_k=5).item()
#     ndcg = retrieval_normalized_dcg(preds=pred, target=target, top_k=10).item()
    
#     r5s.append(r5)
#     mrrs.append(mrr)
#     rmaps.append(rmap)
#     ndcgs.append(ndcg)

# print('mean recall5@891: ', mean(r5s))
# print('mean mrr: ', mean(mrrs))
# print('mean rmAP: ', mean(rmaps))
# print('mean ndcg10@891: ', mean(ndcgs))

In [44]:
from statistics import mean

import torch

from torchmetrics.functional.retrieval import (
    retrieval_recall,
    retrieval_reciprocal_rank,
    retrieval_average_precision,
    retrieval_normalized_dcg,
)

# Проверяем, что alignment сохранён
assert len(queries) == len(docs), "Длины queries и docs должны совпадать."

n = len(docs)

print("Number of retrieval queries:", n)
print("Number of retrieval docs:", n)

if n == 0:
    retrieval_metrics = {
        "recall@5": 0.0,
        "mrr": 0.0,
        "map@5": 0.0,
        "ndcg@10": 0.0,
        "num_queries": 0,
        "num_docs": 0,
    }
else:
    r5s = []
    mrrs = []
    rmaps = []
    ndcgs = []

    top_k_5 = min(5, n)
    top_k_10 = min(10, n)

    for i in range(n):
        pred = build_pred_mask_for_i(cos_scores, i=i, n=n)
        target = build_target_mask_for_i(i=i, n=n)

        pred = torch.tensor(pred, dtype=torch.float32)
        target = torch.tensor(target, dtype=torch.bool)

        r5 = retrieval_recall(
            preds=pred,
            target=target,
            top_k=top_k_5,
        ).item()

        mrr = retrieval_reciprocal_rank(
            preds=pred,
            target=target,
            top_k=top_k_5,
        ).item()

        rmap = retrieval_average_precision(
            preds=pred,
            target=target,
            top_k=top_k_5,
        ).item()

        ndcg = retrieval_normalized_dcg(
            preds=pred,
            target=target,
            top_k=top_k_10,
        ).item()

        r5s.append(r5)
        mrrs.append(mrr)
        rmaps.append(rmap)
        ndcgs.append(ndcg)

    retrieval_metrics = {
        "recall@5": float(mean(r5s)),
        "mrr": float(mean(mrrs)),
        "map@5": float(mean(rmaps)),
        "ndcg@10": float(mean(ndcgs)),
        "num_queries": int(n),
        "num_docs": int(n),
    }

print()
print(f"mean recall@5@{n}: {retrieval_metrics['recall@5']:.4f}")
print(f"mean mrr:          {retrieval_metrics['mrr']:.4f}")
print(f"mean map@5:        {retrieval_metrics['map@5']:.4f}")
print(f"mean ndcg@10@{n}: {retrieval_metrics['ndcg@10']:.4f}")

Number of retrieval queries: 886
Number of retrieval docs: 886

mean recall@5@886: 0.3950
mean mrr:          0.3148
mean map@5:        0.3148
mean ndcg@10@886: 0.3492


In [9]:
from statistics import mean

import torch

from sentence_transformers import util

from torchmetrics.functional.retrieval import (
    retrieval_recall,
    retrieval_reciprocal_rank,
    retrieval_average_precision,
    retrieval_normalized_dcg,
)


def normalize_text_for_eval(text):
    return " ".join(str(text).split())


def compute_retrieval_metrics(
    model,
    queries,
    docs,
    query_prefix="",
    doc_prefix="",
    batch_size=16,
    device=None,
    score_top_k=100,
    ndcg_top_k=10,
    normalize_embeddings=True,
    show_progress_bar=False,
    max_seq_length=None,
):
    """
    Считает retrieval metrics:
        recall@5
        mrr
        map@5
        ndcg@10

    Ожидается, что queries[i] соответствует правильному docs[i].
    """
    n_docs = len(docs)
    n_queries = len(queries)

    if n_queries != n_docs:
        raise ValueError(
            "Для этой функции retrieval-метрики предполагают, что "
            "len(queries) == len(docs), и queries[i] соответствует docs[i]."
        )

    if n_docs == 0:
        return {
            "recall@5": 0.0,
            "mrr": 0.0,
            "map@5": 0.0,
            "ndcg@10": 0.0,
            "num_queries": 0,
            "num_docs": 0,
        }

    model.eval()

    if max_seq_length is not None and hasattr(model, "max_seq_length"):
        model.max_seq_length = max_seq_length

    if device is None:
        try:
            device = next(model.parameters()).device
        except Exception:
            device = "cpu"

    prefixed_queries = [
        f"{query_prefix}{normalize_text_for_eval(q)}"
        for q in queries
    ]

    prefixed_docs = [
        f"{doc_prefix}{normalize_text_for_eval(d)}"
        for d in docs
    ]

    with torch.no_grad():
        query_embs = model.encode(
            prefixed_queries,
            convert_to_tensor=True,
            device=device,
            batch_size=batch_size,
            normalize_embeddings=normalize_embeddings,
            show_progress_bar=show_progress_bar,
        )

        doc_embs = model.encode(
            prefixed_docs,
            convert_to_tensor=True,
            device=device,
            batch_size=batch_size,
            normalize_embeddings=normalize_embeddings,
            show_progress_bar=show_progress_bar,
        )

        top_k_scores = min(score_top_k, n_docs)

        cos_scores = util.semantic_search(
            query_embs,
            doc_embs,
            top_k=top_k_scores,
        )

    if len(cos_scores) != n_queries:
        raise ValueError(
            f"semantic_search вернул {len(cos_scores)} результатов, "
            f"а ожидалось {n_queries}."
        )

    r5_top_k = min(5, n_docs)
    map_top_k = min(5, n_docs)
    ndcg_top_k = min(ndcg_top_k, n_docs)

    recall_scores = []
    mrr_scores = []
    map_scores = []
    ndcg_scores = []

    for i in range(n_queries):
        pred = [0.0] * n_docs

        for hit in cos_scores[i]:
            doc_id = int(hit["corpus_id"])
            score = float(hit["score"])
            pred[doc_id] = score

        target = [False] * n_docs
        target[i] = True

        pred_tensor = torch.tensor(pred, dtype=torch.float32)
        target_bool = torch.tensor(target, dtype=torch.bool)
        target_rel = torch.tensor(target, dtype=torch.float32)

        recall_5 = retrieval_recall(
            preds=pred_tensor,
            target=target_bool,
            top_k=r5_top_k,
        ).item()

        mrr = retrieval_reciprocal_rank(
            preds=pred_tensor,
            target=target_bool,
            top_k=r5_top_k,
        ).item()

        rmap = retrieval_average_precision(
            preds=pred_tensor,
            target=target_bool,
            top_k=map_top_k,
        ).item()

        ndcg = retrieval_normalized_dcg(
            preds=pred_tensor,
            target=target_rel,
            top_k=ndcg_top_k,
        ).item()

        recall_scores.append(recall_5)
        mrr_scores.append(mrr)
        map_scores.append(rmap)
        ndcg_scores.append(ndcg)

    return {
        "recall@5": float(mean(recall_scores)),
        "mrr": float(mean(mrr_scores)),
        "map@5": float(mean(map_scores)),
        "ndcg@10": float(mean(ndcg_scores)),
        "num_queries": int(n_queries),
        "num_docs": int(n_docs),
    }


print("compute_retrieval_metrics создана")

compute_retrieval_metrics создана


In [10]:
import os
import gc
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

# ============================================================================
# 0. Проверка, что fine-tuned checkpoint действительно существует
# ============================================================================

OUTPUT_DIR = globals().get("OUTPUT_DIR", "results/e5_sberquad_finetuned")

MODEL_NAME = globals().get("MODEL_NAME", "intfloat/multilingual-e5-small")
MAX_LEN = globals().get("MAX_LEN", 512)
DEVICE = globals().get("DEVICE", "cpu")

QUERY_PREFIX = globals().get("QUERY_PREFIX", "query: ")
PASSAGE_PREFIX = globals().get("PASSAGE_PREFIX", "passage: ")

if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Не найден fine-tuned checkpoint в папке: {OUTPUT_DIR}. "
        f"Проверь, что в VSCode путь notebook к папке results/e5_sberquad_finetuned правильный."
    )

print("OUTPUT_DIR exists:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("MAX_LEN:", MAX_LEN)


# ============================================================================
# 1. Если evaluator_test еще не определен, создаем его из test_shuffled_data()
# ============================================================================

if "evaluator_test" not in globals():
    print("evaluator_test не найден, создаем его...")

    from sentence_transformers import evaluation

    n_test_samples = globals().get("n_test_samples", 1000)

    test = test_shuffled_data()
    test_sample = test[:n_test_samples]

    test_questions_raw, test_document_contexts_raw, test_coss = zip(*test_sample)

    if "multilingual-e5" in MODEL_NAME:
        test_questions = [QUERY_PREFIX + q for q in test_questions_raw]
        test_document_contexts = [PASSAGE_PREFIX + d for d in test_document_contexts_raw]
    else:
        test_questions = test_questions_raw
        test_document_contexts = test_document_contexts_raw

    evaluator_test = evaluation.EmbeddingSimilarityEvaluator(
        test_questions,
        test_document_contexts,
        test_coss,
    )

    print("evaluator_test создан.")


# ============================================================================
# 2. Готовим positive-only retrieval dataset
# ============================================================================

def prepare_positive_retrieval_dataset(
    max_samples: int = 1000,
    seed: int = 42,
):
    """
    Возвращает только positive QA-пары и убирает дубликаты документов.

    После этого считается, что queries[i] соответствует docs[i].
    """
    test = test_shuffled_data()
    sample = test[:max_samples]

    seen_docs = set()

    queries = []
    docs = []

    for q, d, label in sample:
        if float(label) <= 0.5:
            continue

        q = " ".join(str(q).split())
        d = " ".join(str(d).split())

        if not q or not d:
            continue

        if d in seen_docs:
            continue

        seen_docs.add(d)
        queries.append(q)
        docs.append(d)

    return queries, docs


retrieval_query_prefix = QUERY_PREFIX if "multilingual-e5" in MODEL_NAME else ""
retrieval_doc_prefix = PASSAGE_PREFIX if "multilingual-e5" in MODEL_NAME else ""

print("Preparing positive retrieval dataset...")

retrieval_queries, retrieval_docs = prepare_positive_retrieval_dataset(
    max_samples=1000,
    seed=42,
)

print("Retrieval queries:", len(retrieval_queries))
print("Retrieval docs:", len(retrieval_docs))


# ============================================================================
# 3. Утилита для освобождения памяти
# ============================================================================

def free_model(model):
    """
    Пытаемся освободить память GPU после оценки модели.
    """
    try:
        model.to("cpu")
    except Exception:
        pass

    try:
        del model
    except Exception:
        pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# 4. Оценка одной модели
# ============================================================================

def evaluate_model_for_stage(model, stage_name):
    result = {
        "stage": stage_name,
        "model_path": stage_name,
    }

    print("\n" + "=" * 80)
    print(f"Evaluating: {stage_name}")
    print("=" * 80)

    # Pair similarity metrics: pearson / spearman
    try:
        pair_metrics = evaluator_test(model)
        result.update({
            f"pair_{k}": float(v)
            for k, v in pair_metrics.items()
        })
        print("Pair metrics:")
        print(pair_metrics)
    except Exception as e:
        print("Pair metrics failed:", e)
        result["pair_pearson_cosine"] = None
        result["pair_spearman_cosine"] = None

    # Retrieval metrics: recall@5, mrr, map@5, ndcg@10
    if "compute_retrieval_metrics" not in globals():
        print("compute_retrieval_metrics не найден, пропускаем retrieval metrics.")
        result.update({
            "recall@5": None,
            "mrr": None,
            "map@5": None,
            "ndcg@10": None,
        })
    else:
        try:
            retrieval_metrics = compute_retrieval_metrics(
                model=model,
                queries=retrieval_queries,
                docs=retrieval_docs,
                query_prefix=retrieval_query_prefix,
                doc_prefix=retrieval_doc_prefix,
                batch_size=16,
                max_seq_length=MAX_LEN,
                show_progress_bar=True,
            )

            result.update({
                "recall@5": float(retrieval_metrics["recall@5"]),
                "mrr": float(retrieval_metrics["mrr"]),
                "map@5": float(retrieval_metrics["map@5"]),
                "ndcg@10": float(retrieval_metrics["ndcg@10"]),
                "num_retrieval_queries": int(retrieval_metrics["num_queries"]),
                "num_retrieval_docs": int(retrieval_metrics["num_docs"]),
            })

            print("Retrieval metrics:")
            print(retrieval_metrics)

        except Exception as e:
            print("Retrieval metrics failed:", e)
            result.update({
                "recall@5": None,
                "mrr": None,
                "map@5": None,
                "ndcg@10": None,
            })

    return result


# ============================================================================
# 5. Оценка baseline и fine-tuned по очереди
# ============================================================================

results = []

# ----------------------------
# Baseline
# ----------------------------
print("\nLoading baseline model...")

baseline_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

baseline_model.max_seq_length = MAX_LEN

baseline_result = evaluate_model_for_stage(
    model=baseline_model,
    stage_name="baseline",
)

results.append(baseline_result)

free_model(baseline_model)

# ----------------------------
# Fine-tuned
# ----------------------------
print("\nLoading fine-tuned model from checkpoint...")

fine_tuned_model = SentenceTransformer(
    OUTPUT_DIR,
    device=DEVICE,
)

fine_tuned_model.max_seq_length = MAX_LEN

fine_tuned_result = evaluate_model_for_stage(
    model=fine_tuned_model,
    stage_name="fine_tuned",
)

results.append(fine_tuned_result)

free_model(fine_tuned_model)


# ============================================================================
# 6. Итоговая таблица
# ============================================================================

df = pd.DataFrame(results)

display_columns = [
    "stage",
    "pair_pearson_cosine",
    "pair_spearman_cosine",
    "recall@5",
    "mrr",
    "map@5",
    "ndcg@10",
    "num_retrieval_queries",
    "num_retrieval_docs",
]

# Оставляем только существующие колонки
display_columns = [
    col for col in display_columns
    if col in df.columns
]

print("\nFinal comparison table:")

print(df[display_columns].to_string(index=False))

# Сохраняем в CSV
output_csv = "baseline_vs_finetuned_metrics.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Таблица сохранена в: {output_csv}")

OUTPUT_DIR exists: results/e5_sberquad_finetuned
MODEL_NAME: intfloat/multilingual-e5-small
DEVICE: cpu
MAX_LEN: 512
Preparing positive retrieval dataset...
Retrieval queries: 479
Retrieval docs: 479

Loading baseline model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3635.07it/s]



Evaluating: baseline
Pair metrics:
{'pearson_cosine': 0.9201645153699128, 'spearman_cosine': 0.8646678652730023}


Batches: 100%|██████████| 30/30 [01:13<00:00,  2.43s/it]


Retrieval metrics:
{'recall@5': 0.9895615866388309, 'mrr': 0.9472860127127494, 'map@5': 0.9472860127127494, 'ndcg@10': 0.9609033026127826, 'num_queries': 479, 'num_docs': 479}

Loading fine-tuned model from checkpoint...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 513.61it/s]



Evaluating: fine_tuned
Pair metrics:
{'pearson_cosine': 0.9129187049933101, 'spearman_cosine': 0.8514345670144171}


Batches: 100%|██████████| 30/30 [01:12<00:00,  2.42s/it]


Retrieval metrics:
{'recall@5': 0.8559498956158664, 'mrr': 0.6832985394558478, 'map@5': 0.6832985394558478, 'ndcg@10': 0.7471451744406109, 'num_queries': 479, 'num_docs': 479}

Final comparison table:
     stage  pair_pearson_cosine  pair_spearman_cosine  recall@5      mrr    map@5  ndcg@10  num_retrieval_queries  num_retrieval_docs
  baseline             0.920165              0.864668  0.989562 0.947286 0.947286 0.960903                    479                 479
fine_tuned             0.912919              0.851435  0.855950 0.683299 0.683299 0.747145                    479                 479
Таблица сохранена в: baseline_vs_finetuned_metrics.csv


In [12]:
import os
import gc
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

# ============================================================================
# 0. Проверка, что fine-tuned checkpoint действительно существует
# ============================================================================

OUTPUT_DIR = globals().get("OUTPUT_DIR", "results/e5_sberquad_finetuned")

MODEL_NAME = globals().get("MODEL_NAME", "intfloat/multilingual-e5-small")
MAX_LEN = globals().get("MAX_LEN", 512)
DEVICE = globals().get("DEVICE", "cpu")

QUERY_PREFIX = globals().get("QUERY_PREFIX", "query: ")
PASSAGE_PREFIX = globals().get("PASSAGE_PREFIX", "passage: ")

SEED = globals().get("SEED", 42)
N_RETRIEVAL_SAMPLES = globals().get("N_RETRIEVAL_SAMPLES", 1000)

if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Не найден fine-tuned checkpoint в папке: {OUTPUT_DIR}. "
        f"Проверь, что в VSCode путь notebook к папке results/e5_sberquad_finetuned правильный."
    )

print("OUTPUT_DIR exists:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("MAX_LEN:", MAX_LEN)


# ============================================================================
# 1. Если evaluator_test еще не определен, создаем его из test_shuffled_data()
#
# evaluator_test нужен для pair metrics:
#   pearson_cosine
#   spearman_cosine
#
# Для pair metrics negative пары НУЖНЫ.
# ============================================================================

if "evaluator_test" not in globals():
    print("evaluator_test не найден, создаем его...")

    from sentence_transformers import evaluation

    n_test_samples = globals().get("n_test_samples", 1000)

    test = test_shuffled_data()
    test_sample = test[:n_test_samples]

    test_questions_raw, test_document_contexts_raw, test_coss = zip(*test_sample)

    if "multilingual-e5" in MODEL_NAME:
        test_questions = [QUERY_PREFIX + q for q in test_questions_raw]
        test_document_contexts = [PASSAGE_PREFIX + d for d in test_document_contexts_raw]
    else:
        test_questions = test_questions_raw
        test_document_contexts = test_document_contexts_raw

    evaluator_test = evaluation.EmbeddingSimilarityEvaluator(
        test_questions,
        test_document_contexts,
        test_coss,
    )

    print("evaluator_test создан.")


# ============================================================================
# 2. Готовим retrieval datasets
#
# Вариант 1: positive_only
#   Оставляем только positive QA-пары.
#   Это корректный retrieval-датасет.
#
# Вариант 2: mixed
#   Не удаляем negative QA-пары.
#   Это более "как у преподавателя".
#   Но для retrieval это методологически спорный вариант,
#   потому что target документа может быть негативным.
# ============================================================================

def normalize_text_for_eval(text):
    return " ".join(str(text).split())


def prepare_retrieval_dataset(
    max_samples: int = 1000,
    positive_only: bool = True,
    drop_duplicate_docs: bool = True,
    seed: int = SEED,
):
    """
    Готовит retrieval dataset.

    positive_only=True:
        оставляем только пары с label > 0.5.

    positive_only=False:
        оставляем positive и negative пары.

    drop_duplicate_docs=True:
        убираем дубликаты документов, сохраняя первый occurrence.
    """
    test = test_shuffled_data()
    sample = test[:max_samples]

    if positive_only:
        pairs = [
            (q, d)
            for q, d, label in sample
            if float(label) > 0.5
        ]
    else:
        pairs = [
            (q, d)
            for q, d, label in sample
        ]

    queries = []
    docs = []

    if drop_duplicate_docs:
        seen_docs = set()

        for q, d in pairs:
            q = normalize_text_for_eval(q)
            d = normalize_text_for_eval(d)

            if not q or not d:
                continue

            if d in seen_docs:
                continue

            seen_docs.add(d)
            queries.append(q)
            docs.append(d)

        return queries, docs

    for q, d in pairs:
        q = normalize_text_for_eval(q)
        d = normalize_text_for_eval(d)

        if not q or not d:
            continue

        queries.append(q)
        docs.append(d)

    return queries, docs


retrieval_query_prefix = QUERY_PREFIX if "multilingual-e5" in MODEL_NAME else ""
retrieval_doc_prefix = PASSAGE_PREFIX if "multilingual-e5" in MODEL_NAME else ""


print("Preparing positive-only retrieval dataset...")

positive_queries, positive_docs = prepare_retrieval_dataset(
    max_samples=N_RETRIEVAL_SAMPLES,
    positive_only=True,
    drop_duplicate_docs=True,
    seed=SEED,
)

print("Positive-only retrieval queries:", len(positive_queries))
print("Positive-only retrieval docs:", len(positive_docs))


print("Preparing mixed retrieval dataset without removing negatives...")

mixed_queries, mixed_docs = prepare_retrieval_dataset(
    max_samples=N_RETRIEVAL_SAMPLES,
    positive_only=False,
    drop_duplicate_docs=True,
    seed=SEED,
)

print("Mixed retrieval queries:", len(mixed_queries))
print("Mixed retrieval docs:", len(mixed_docs))


retrieval_sets = {
    "positive_only": (positive_queries, positive_docs),
    "mixed": (mixed_queries, mixed_docs),
}


# ============================================================================
# 3. Утилита для освобождения памяти
# ============================================================================

def free_model(model):
    """
    Пытаемся освободить память GPU после оценки модели.
    """
    try:
        model.to("cpu")
    except Exception:
        pass

    try:
        del model
    except Exception:
        pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# 4. Оценка одной модели
# ============================================================================

def empty_retrieval_metrics(prefix: str):
    return {
        f"{prefix}_recall@5": None,
        f"{prefix}_mrr": None,
        f"{prefix}_map@5": None,
        f"{prefix}_ndcg@10": None,
        f"{prefix}_num_queries": None,
        f"{prefix}_num_docs": None,
    }


def evaluate_model_for_stage(model, stage_name):
    result = {
        "stage": stage_name,
        "model_path": stage_name,
    }

    print("\n" + "=" * 80)
    print(f"Evaluating: {stage_name}")
    print("=" * 80)

    # ------------------------------------------------------------------------
    # Pair similarity metrics: pearson / spearman
    # Это метрики для positive/negative пар.
    # Здесь negative удалять не нужно.
    # ------------------------------------------------------------------------
    try:
        pair_metrics = evaluator_test(model)

        result.update({
            f"pair_{k}": float(v)
            for k, v in pair_metrics.items()
        })

        print("Pair metrics:")
        print(pair_metrics)

    except Exception as e:
        print("Pair metrics failed:", e)

        result["pair_pearson_cosine"] = None
        result["pair_spearman_cosine"] = None

    # ------------------------------------------------------------------------
    # Retrieval metrics
    #
    # positive_only:
    #   корректный вариант
    #
    # mixed:
    #   вариант без удаления negatives
    # ------------------------------------------------------------------------
    if "compute_retrieval_metrics" not in globals():
        print("compute_retrieval_metrics не найден, пропускаем retrieval metrics.")

        for prefix in retrieval_sets:
            result.update(empty_retrieval_metrics(prefix))

    else:
        for prefix, (queries, docs) in retrieval_sets.items():
            print(f"\nRetrieval metrics for set: {prefix}")

            try:
                retrieval_metrics = compute_retrieval_metrics(
                    model=model,
                    queries=queries,
                    docs=docs,
                    query_prefix=retrieval_query_prefix,
                    doc_prefix=retrieval_doc_prefix,
                    batch_size=16,
                    max_seq_length=MAX_LEN,
                    show_progress_bar=True,
                )

                result.update({
                    f"{prefix}_recall@5": float(retrieval_metrics["recall@5"]),
                    f"{prefix}_mrr": float(retrieval_metrics["mrr"]),
                    f"{prefix}_map@5": float(retrieval_metrics["map@5"]),
                    f"{prefix}_ndcg@10": float(retrieval_metrics["ndcg@10"]),
                    f"{prefix}_num_queries": int(retrieval_metrics["num_queries"]),
                    f"{prefix}_num_docs": int(retrieval_metrics["num_docs"]),
                })

                print(retrieval_metrics)

            except Exception as e:
                print(f"Retrieval metrics failed for {prefix}:", e)
                result.update(empty_retrieval_metrics(prefix))

    return result


# ============================================================================
# 5. Оценка baseline и fine-tuned по очереди
# ============================================================================

results = []

# ----------------------------
# Baseline
# ----------------------------
print("\nLoading baseline model...")

baseline_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

baseline_model.max_seq_length = MAX_LEN

baseline_result = evaluate_model_for_stage(
    model=baseline_model,
    stage_name="baseline",
)

results.append(baseline_result)

free_model(baseline_model)

# ----------------------------
# Fine-tuned
# ----------------------------
print("\nLoading fine-tuned model from checkpoint...")

fine_tuned_model = SentenceTransformer(
    OUTPUT_DIR,
    device=DEVICE,
)

fine_tuned_model.max_seq_length = MAX_LEN

fine_tuned_result = evaluate_model_for_stage(
    model=fine_tuned_model,
    stage_name="fine_tuned",
)

results.append(fine_tuned_result)

free_model(fine_tuned_model)


# ============================================================================
# 6. Итоговая таблица
# ============================================================================

df = pd.DataFrame(results)

display_columns = [
    "stage",

    "pair_pearson_cosine",
    "pair_spearman_cosine",

    "positive_only_recall@5",
    "positive_only_mrr",
    "positive_only_map@5",
    "positive_only_ndcg@10",
    "positive_only_num_queries",
    "positive_only_num_docs",

    "mixed_recall@5",
    "mixed_mrr",
    "mixed_map@5",
    "mixed_ndcg@10",
    "mixed_num_queries",
    "mixed_num_docs",
]

# Оставляем только существующие колонки
display_columns = [
    col for col in display_columns
    if col in df.columns
]

print("\nFinal comparison table:")
print(df[display_columns].to_string(index=False))

# Сохраняем в CSV
output_csv = "baseline_vs_finetuned_metrics_positive_and_mixed.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Таблица сохранена в: {output_csv}")

OUTPUT_DIR exists: results/e5_sberquad_finetuned
MODEL_NAME: intfloat/multilingual-e5-small
DEVICE: cpu
MAX_LEN: 512
Preparing positive-only retrieval dataset...
Positive-only retrieval queries: 479
Positive-only retrieval docs: 479
Preparing mixed retrieval dataset without removing negatives...
Mixed retrieval queries: 886
Mixed retrieval docs: 886

Loading baseline model...


Loading weights: 100%|██████████| 199/199 [00:28<00:00,  6.88it/s]



Evaluating: baseline
Pair metrics:
{'pearson_cosine': 0.9201645153699128, 'spearman_cosine': 0.8646678652730023}

Retrieval metrics for set: positive_only


Batches: 100%|██████████| 30/30 [01:14<00:00,  2.47s/it]


{'recall@5': 0.9895615866388309, 'mrr': 0.9472860127127494, 'map@5': 0.9472860127127494, 'ndcg@10': 0.9609033026127826, 'num_queries': 479, 'num_docs': 479}

Retrieval metrics for set: mixed


...


Evaluating: fine_tuned
Pair metrics:
{'pearson_cosine': 0.9129187049933101, 'spearman_cosine': 0.8514345670144171}

Retrieval metrics for set: positive_only


Batches: 100%|██████████| 30/30 [01:13<00:00,  2.44s/it]


{'recall@5': 0.8559498956158664, 'mrr': 0.6832985394558478, 'map@5': 0.6832985394558478, 'ndcg@10': 0.7471451744406109, 'num_queries': 479, 'num_docs': 479}

Retrieval metrics for set: mixed


Batches: 100%|██████████| 56/56 [02:13<00:00,  2.38s/it]


{'recall@5': 0.39503386004514673, 'mrr': 0.31480436459284605, 'map@5': 0.31480436459284605, 'ndcg@10': 0.3491692634522242, 'num_queries': 886, 'num_docs': 886}

Final comparison table:
     stage  pair_pearson_cosine  pair_spearman_cosine  positive_only_recall@5  positive_only_mrr  positive_only_map@5  positive_only_ndcg@10  positive_only_num_queries  positive_only_num_docs  mixed_recall@5  mixed_mrr  mixed_map@5  mixed_ndcg@10  mixed_num_queries  mixed_num_docs
  baseline             0.920165              0.864668                0.989562           0.947286             0.947286               0.960903                        479                     479        0.498871   0.469263     0.469263       0.478500                886             886
fine_tuned             0.912919              0.851435                0.855950           0.683299             0.683299               0.747145                        479                     479        0.395034   0.314804     0.314804       0.349169     

**Вывод:** Для выбранного QA/retrieval-датасета исходная модель multilingual-e5-small уже демонстрирует очень высокое качество поиска документов. Текущая схема fine-tuning не дала улучшения и снизила метрики, поэтому для данной задачи целесообразно использовать baseline модель либо применять более правильный retrieval-ориентированный подход к дообучению.